# Question 25 - 125. Valid Palindrome

A phrase is a **palindrome** if, after converting all uppercase letters into lowercase letters and removing all non-alphanumeric characters, it reads the same forward and backward. Alphanumeric characters include letters and numbers.

Given a string `s`, return `true` *if it is a **palindrome**, or* `false` *otherwise*.

**Example 1:**

    Input: s = "A man, a plan, a canal: Panama"
    Output: true
    Explanation: "amanaplanacanalpanama" is a palindrome.

**Example 2:**

    Input: s = "race a car"
    Output: false
    Explanation: "raceacar" is not a palindrome.

**Example 3:**

    Input: s = " "
    Output: true
    Explanation: s is an empty string "" after removing non-alphanumeric characters.
    Since an empty string reads the same forward and backward, it is a palindrome.

**Constraints:**

- `1 <= s.length <= 2 * 10^5`
- `s` consists only of printable ASCII characters.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers converging from both ends (skip non-alphanumerics on the fly)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (clean copy + reverse) | O(n) | O(n) |
| 2️⃣ Optimized (two pointers in place) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

A palindrome is something that reads the same forwards and backwards, like **"racecar"** or **"noon"**.

Here there's a twist: we're told to **ignore punctuation and spaces**, and to **treat `A` and `a` as the same**. So the famous sentence

> "A man, a plan, a canal: Panama"

becomes `amanaplanacanalpanama` once you strip the clutter, and that *does* read the same both ways.

**Everyday picture:** imagine the sentence printed on a strip of paper. Put your **left index finger on the first letter** and your **right index finger on the last letter**. Compare the two letters. If they match, move both fingers one step toward the middle and compare again. If your finger lands on a comma or a space, just slide past it, because it doesn't count. If you ever find two letters that don't match, it's not a palindrome. If your fingers meet in the middle without a mismatch, it is.

That finger-walking idea *is* the optimal algorithm. The rest of this notebook is about getting there honestly and explaining why it's the best choice.

### Step 0 · Clarify before coding

Before writing any code, confirm the rules so you don't solve the wrong problem.

🗣️ *"Let me make sure I understand the rules. I only care about letters and digits. Spaces, commas, colons and so on are ignored. Uppercase and lowercase count as the same letter. Then the question is whether that cleaned-up sequence reads the same forwards and backwards. Is that right?"*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Do **digits** count? | **Yes**, they're alphanumeric. | `"0P"` → `0` vs `p` → **not** a palindrome. Easy to get wrong. |
| What if nothing is left after cleaning (e.g. `" "`)? | Empty counts as a palindrome → `true`. | Our code must not crash on "nothing to compare". |
| What characters can appear? | Printable ASCII. | So Python's `isalnum()` and `lower()` behave predictably. |
| How long can it be? | Up to 200,000 characters. | Anything O(n²) is risky. We want one pass. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** do exactly what the problem statement says, literally.
1. Make a **cleaned copy**: keep only letters and digits, all lowercased.
2. Make a **reversed copy** of that.
3. Check whether the two copies are identical.

🗣️ *"My first instinct is to follow the definition directly. I'll build a new string with just the letters and digits in lowercase, then compare it to its reverse. If they're equal, it's a palindrome. That's easy to get right, which is why I start with it."*

**Is it slow?** Not really. Each step walks the string once, so the time grows in a straight line with the length (that's what **O(n)** means: double the input, double the work).

**So what's wrong with it?** It's **wasteful with memory**. We build two brand-new strings, each potentially as long as the input, just to compare letters that were already sitting in the original string.

🗣️ *"This is O(n) time, but it also uses O(n) extra memory for the cleaned copy and the reversed copy. The interviewer will likely ask if I can avoid making copies, so let me think about that."*

In [1]:
class SolutionBrute:
    def isPalindrome(self, s: str) -> bool:
        cleaned = "".join(ch.lower() for ch in s if ch.isalnum())  # copy #1
        return cleaned == cleaned[::-1]                            # copy #2 (reversed)

### Step 2 · Optimize: the "aha" moment

**What are we really doing when we compare a string with its reverse?**

We're checking: *first letter == last letter? second == second-to-last? third == third-to-last? …*

That's just **pairs from the outside moving in**. We don't need copies for that. We can read the pairs **directly from the original string**, using two position markers (the "fingers" from the intro).

> 💡 **Two pointers** is just a fancy name for "two index variables walking through the data". Here one starts at the **left end** and one at the **right end**, and they walk **toward each other**.

**The algorithm in plain words:**
1. `left` starts at the first character, `right` at the last.
2. If `left` is on junk (a space or punctuation), step it right. If `right` is on junk, step it left.
3. Now both are on real characters, so compare them, ignoring case.
   - Different → **not** a palindrome, stop immediately.
   - Same → move both one step inward.
4. When the pointers meet or cross, every pair has matched → **palindrome**.

🗣️ *"Comparing a string with its reverse is really just comparing pairs from the outside in, so I can do that directly on the original string with two indexes, one starting at each end. Each index skips anything that isn't a letter or digit. When both are on real characters, I compare them in lowercase. If they differ, I return false right away. If they match, I move both inward. When they meet, I return true. I never build a new string, so the extra memory is constant, just two integers, and each character is looked at once, so it's still linear time."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Clean + reverse + compare | ✅ fine first answer | Easy to read and correct, but it makes two extra copies of the text. |
| Regex to clean the string | ⚠️ | Still makes a copy, and regex adds overhead and room for mistakes. No real benefit. |
| Use a **stack** (push the first half, pop to compare) | ❌ | A stack is "last in, first out", like a pile of plates. It *can* reverse things, but it still stores half the string, so no memory saving. More code, no gain. |
| Recursion: check ends, then recurse on the middle | ❌ | Each recursive call slices a new middle string (copying again and again → O(n²)), and 200,000 nested calls would overflow Python's call stack. |
| **Two pointers, skipping junk in place** | ✅ **best** | Same speed as the obvious way, but **zero copies**. It also stops early at the first mismatch. |

**The one-line takeaway:** *when a problem compares "the front" with "the back", think two pointers moving inward.*

In [2]:
class Solution:
    def isPalindrome(self, s: str) -> bool:
        left, right = 0, len(s) - 1
        while left < right:
            # slide past anything that isn't a letter/digit
            while left < right and not s[left].isalnum():
                left += 1
            while left < right and not s[right].isalnum():
                right -= 1
            # compare the pair, ignoring case
            if s[left].lower() != s[right].lower():
                return False
            # this pair is fine; move both fingers inward
            left += 1
            right -= 1
        return True

### Step 3 · Toy example walkthrough (tell it like a story)

**Example:** `s = "race a car"`. Positions are numbered 0 to 9:

```
index:  0 1 2 3 4 5 6 7 8 9
char:   r a c e _ a _ c a r      (_ = space)
```

| step | left → char | right → char | what happens |
|---|---|---|---|
| 1 | 0 → `r` | 9 → `r` | `r` = `r` ✅ move both inward |
| 2 | 1 → `a` | 8 → `a` | `a` = `a` ✅ move inward |
| 3 | 2 → `c` | 7 → `c` | `c` = `c` ✅ move inward |
| 4 | 3 → `e` | 6 → *space* | right is on junk → slide right to 5 → `a` |
| 5 | 3 → `e` | 5 → `a` | `e` ≠ `a` ❌ → **return False** |

**Narrated:** "The outer letters r–r, a–a, c–c all match. Then my right finger hits a space, so I skip it and land on 'a'. Now I'm comparing 'e' with 'a', which don't match, so it's not a palindrome. I can stop right here without looking at anything else."

**Second example:** `"A man, a plan, a canal: Panama"`. The pointers glide past every comma, colon and space, compare `A` with `a` (equal after lowercasing), and keep matching until they meet in the middle → **True**.

**A subtle detail worth saying out loud:** the inner skip loops also check `left < right`. Without that guard, a string of pure punctuation like `".,"` could make a pointer run past the other one, or even off the end of the string.

In [3]:
cases = [
    ("A man, a plan, a canal: Panama", True),
    ("race a car", False),
    (" ", True),            # nothing left after cleaning
    ("0P", False),          # digits count!
    ("a.", True),
    (".,", True),           # only punctuation
    ("ab_a", True),         # '_' is not alphanumeric
    ("Was it a car or a cat I saw?", True),
]
for s, expected in cases:
    assert SolutionBrute().isPalindrome(s) == expected
    assert Solution().isPalindrome(s) == expected
    print(repr(s), "->", expected)

# random cross-check: optimized must always agree with the obvious version
import random
for _ in range(2000):
    s = "".join(random.choices("aAb1 ,.", k=random.randint(1, 8)))
    assert Solution().isPalindrome(s) == SolutionBrute().isPalindrome(s), s
print("All tests passed ✅")

'A man, a plan, a canal: Panama' -> True
'race a car' -> False
' ' -> True
'0P' -> False
'a.' -> True
'.,' -> True
'ab_a' -> True
'Was it a car or a cat I saw?' -> True
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Clean + reverse | O(n) | O(n) | Reads the text a few times and makes two copies of it. |
| **Two pointers** | **O(n)** | **O(1)** | Reads each character at most once and stores only two numbers. |

**Why is the two-pointer version O(n) time, even with loops inside loops?**
It *looks* like nested loops, but notice that `left` **only ever moves right** and `right` **only ever moves left**, and everything stops when they meet. So between them they take at most n steps in total. Nested-looking code isn't automatically O(n²). What matters is how many total steps the pointers can take.

**Why O(1) memory?** No matter whether the input has 10 characters or 200,000, we only store two integers (`left` and `right`). The memory doesn't grow with the input.

**Edge cases to mention:**
- Only punctuation or spaces → `True` (nothing to disagree).
- Digits mixed with letters → `"0P"` is `False`.
- Single character → `True`.
- Mixed case → `"Aa"` is `True`.

**Likely follow-up:** *"What if you're allowed to delete at most one character?"* (LC 680). Use the same two pointers. At the first mismatch, try skipping the left character **or** the right character, and check whether the remaining middle is a palindrome. Still O(n).

---

#### 🗣️ Full interview script (about 60 seconds)

*"A palindrome reads the same forwards and backwards. Here I ignore anything that isn't a letter or digit and compare case-insensitively.*

*The straightforward approach is to build a cleaned lowercase copy and compare it to its reverse. That's linear time but uses linear extra memory for the copies.*

*But comparing a string with its reverse really just means comparing pairs from the outside in: first with last, second with second-to-last, and so on. So I'll use two pointers, one at each end. Each pointer skips non-alphanumeric characters. When both are on real characters, I compare them in lowercase. If they differ, I return false immediately. If they match, I move both inward. When they meet, every pair matched, so I return true.*

*The left pointer only moves right and the right pointer only moves left, so together they take at most n steps, which is O(n) time. I only store two indexes, so the extra space is O(1)."*